In [1]:
import os

for root, dirs, files in os.walk('./listenbrainz_raw/'):
    for f in files:
        path = os.path.join(root, f)
        size = os.path.getsize(path) / 1024 / 1024
        print(f"  {path} ({size:.1f} MB)")

  ./listenbrainz_raw/COPYING (0.0 MB)
  ./listenbrainz_raw/END_TIMESTAMP (0.0 MB)
  ./listenbrainz_raw/SCHEMA_SEQUENCE (0.0 MB)
  ./listenbrainz_raw/START_TIMESTAMP (0.0 MB)
  ./listenbrainz_raw/listens\2026\5.listens (2624.0 MB)


In [2]:
# peek first few lines to identify format
with open('./listenbrainz_raw/listens/2026/5.listens', 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        print(line[:300])
        if i >= 4: break

{"user_id":112348,"user_name":"kojii","timestamp":1779148694,"track_metadata":{"track_name":"Beautiful","artist_name":"kygo and sandro cavazza","release_name":"Golden Hour","additional_info":{"duration":218,"origin_url":"https://music.youtube.com/watch?v=oar7lweDBXA&list=RDAO1kY-3yjsMnkyja8dP5S1kQ",
{"user_id":123061,"user_name":"SonicCD","timestamp":1779148803,"track_metadata":{"track_name":"A Whole New World","artist_name":"Alan Menken","release_name":"Aladdin: Original Motion Picture Soundtrack","additional_info":{"duration_ms":160910,"tracknumber":9,"artist_mbids":["817c68f4-e110-4529-82fa-
{"user_id":119144,"user_name":"rhinowang","timestamp":1779148671,"track_metadata":{"track_name":"Constantinople","artist_name":"Chapelier Fou","release_name":"Méridiens","additional_info":{"recording_msid":"2c9a87f5-49b5-452d-a41b-47bf6afa1ca0"}},"recording_msid":"2c9a87f5-49b5-452d-a41b-47bf6afa1ca
{"user_id":32635,"user_name":"SqueakyBeaverrr","timestamp":1779148696,"track_metadata":{"track_na

In [3]:
# script1b.py
import json
import pandas as pd

src = './listenbrainz_raw/listens/2026/5.listens'
out = './mbdump_small/'

N_USERS   = 1000
MAX_LINES = 5_000_000

# === PASS 1 — collect top N users by play count ===
print("Pass 1 — counting users...")
user_counts = {}
with open(src, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i % 500_000 == 0: print(f"  {i:,} lines scanned")
        if i >= MAX_LINES: break
        try:
            obj = json.loads(line)
            uid = obj['user_id']
            user_counts[uid] = user_counts.get(uid, 0) + 1
        except: continue

top_users = set(
    sorted(user_counts, key=user_counts.get, reverse=True)[:N_USERS]
)
print(f"Top {N_USERS} users selected. Min plays: {sorted(user_counts.values(), reverse=True)[N_USERS-1]}")

# === PASS 1.5 — collect ALL mbids from ALL users ===
print("Pass 1.5 — collecting all mbids from full file...")
all_mbids = set()
with open(src, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i % 500_000 == 0: print(f"  {i:,} scanned | {len(all_mbids):,} mbids found")
        try:
            obj  = json.loads(line)
            mbid = obj.get('track_metadata', {}).get('additional_info', {}).get('recording_mbid')
            if mbid: all_mbids.add(mbid)
        except: continue

pd.Series(list(all_mbids)).to_csv(f'{out}lb_recording_mbids.tsv', index=False, header=False)
print(f"Total unique mbids → {len(all_mbids):,} saved → lb_recording_mbids.tsv")

# === PASS 2 — collect listens for top N users ===
print("Pass 2 — extracting listens...")
rows = []
with open(src, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i % 500_000 == 0: print(f"  {i:,} lines scanned | {len(rows):,} kept")
        if i >= MAX_LINES: break
        try:
            obj  = json.loads(line)
            uid  = obj['user_id']
            if uid not in top_users: continue

            meta = obj.get('track_metadata', {})
            info = meta.get('additional_info', {})

            rows.append({
                'user_id':        uid,
                'user_name':      obj.get('user_name'),
                'timestamp':      obj.get('timestamp'),
                'track_name':     meta.get('track_name'),
                'artist_name':    meta.get('artist_name'),
                'release_name':   meta.get('release_name'),
                'recording_mbid': info.get('recording_mbid'),
                'duration_ms':    info.get('duration_ms') or info.get('duration', 0) * 1000,
            })
        except: continue

history = pd.DataFrame(rows)
print(f"\nDone → {len(history):,} listens | {history['user_id'].nunique()} users")
print(f"With recording_mbid: {history['recording_mbid'].notna().sum():,} rows")

# === SAVE ===
history.to_csv(f'{out}listening_history_real.tsv', sep='\t', index=False)
print(f"Saved → {out}listening_history_real.tsv")
print(history.head(3).to_string())

Pass 1 — counting users...
  0 lines scanned
  500,000 lines scanned
  1,000,000 lines scanned
  1,500,000 lines scanned
  2,000,000 lines scanned
  2,500,000 lines scanned
  3,000,000 lines scanned
  3,500,000 lines scanned
  4,000,000 lines scanned
Top 1000 users selected. Min plays: 133
Pass 1.5 — collecting all mbids from full file...
  0 scanned | 0 mbids found
  500,000 scanned | 4,823 mbids found
  1,000,000 scanned | 13,951 mbids found
  1,500,000 scanned | 23,662 mbids found
  2,000,000 scanned | 44,202 mbids found
  2,500,000 scanned | 55,801 mbids found
  3,000,000 scanned | 64,371 mbids found
  3,500,000 scanned | 73,705 mbids found
  4,000,000 scanned | 78,304 mbids found
Total unique mbids → 81,728 saved → lb_recording_mbids.tsv
Pass 2 — extracting listens...
  0 lines scanned | 0 kept
  500,000 lines scanned | 473,549 kept
  1,000,000 lines scanned | 914,223 kept
  1,500,000 lines scanned | 1,359,357 kept
  2,000,000 lines scanned | 1,738,185 kept
  2,500,000 lines scann

In [4]:
import pandas as pd
import os

OPTS = dict(sep='\t', header=None, engine='python', on_bad_lines='skip', quoting=3)
out  = './mbdump_small/'
os.makedirs(out, exist_ok=True)

def filter_csv(path, names, filter_col, filter_ids, chunksize=50000):
    chunks = []
    for chunk in pd.read_csv(path, names=names, chunksize=chunksize, **OPTS):
        filtered = chunk[chunk[filter_col].isin(filter_ids)]
        if len(filtered): chunks.append(filtered)
    return pd.concat(chunks) if chunks else pd.DataFrame(columns=names)

In [5]:
lb_mbids = set(pd.read_csv(f'{out}lb_recording_mbids.tsv', header=None)[0].dropna())
print(f"mbids to match → {len(lb_mbids):,}")

chunks = []
for chunk in pd.read_csv('./mbdump/mbdump/recording', chunksize=50000,
    names=['id','gid','name','artist_credit','length','comment',
           'edits_pending','last_updated','video'], **OPTS):
    filtered = chunk[chunk['gid'].isin(lb_mbids)]
    if len(filtered): chunks.append(filtered)

recording = pd.concat(chunks) if chunks else pd.DataFrame()
recording = recording[
        recording['name'].notna() & 
        (recording['name'].str.strip() != '') & 
        (~recording['name'].str.lower().isin(['unknown', '[unknown]']))
    ]
    
recording_ids     = set(recording['id'])
artist_credit_ids = set(recording['artist_credit'])
print(f"recording → {len(recording):,} rows matched")

mbids to match → 81,728
recording → 79,359 rows matched


In [6]:
recording_ids     = set(recording['id'])
artist_credit_ids = set(recording['artist_credit'])
artist_credit = filter_csv('./mbdump/mbdump/artist_credit',
    names=['id','name','artist_count','ref_count','created','edits_pending','gid'],
    filter_col='id', filter_ids=artist_credit_ids)

artist_credit_name = filter_csv('./mbdump/mbdump/artist_credit_name',
    names=['artist_credit','position','artist','name','join_phrase'],
    filter_col='artist_credit', filter_ids=artist_credit_ids)
artist_ids = set(artist_credit_name['artist'])

artist = filter_csv('./mbdump/mbdump/artist',
        names=['id','gid','name','sort_name','begin_date_year','begin_date_month',
               'begin_date_day','end_date_year','end_date_month','end_date_day',
               'type','area','gender','comment','edits_pending','last_updated',
               'ended','begin_area','end_area'],
        filter_col='id', filter_ids=artist_ids)

artist = artist[
    artist['name'].notna() & 
    (artist['name'].str.strip() != '') & 
    (~artist['name'].str.lower().isin(['unknown', '[unknown]']))
]
print(f"artist_credit → {len(artist_credit)} | artist_credit_name → {len(artist_credit_name)} | artist → {len(artist)}")

artist_credit → 23828 | artist_credit_name → 35340 | artist → 22973


In [7]:
recording_tag = filter_csv('./mbdump-derived/mbdump/recording_tag',
    names=['recording','tag','count','last_updated'],
    filter_col='recording', filter_ids=recording_ids)
tag_ids = set(recording_tag['tag'])

artist_tag = filter_csv('./mbdump-derived/mbdump/artist_tag',
    names=['artist','tag','count','last_updated'],
    filter_col='artist', filter_ids=artist_ids)
tag_ids.update(artist_tag['tag'])

tag = filter_csv('./mbdump-derived/mbdump/tag',
    names=['id','name','ref_count'],
    filter_col='id', filter_ids=tag_ids)

print(f"recording_tag → {len(recording_tag)} | artist_tag → {len(artist_tag)} | tag → {len(tag)}")

recording_tag → 219438 | artist_tag → 74255 | tag → 9584


In [8]:
chunks = []
for chunk in pd.read_csv('./mbdump/mbdump/l_recording_recording', chunksize=50000,
    names=['id','link','entity0','entity1','edits_pending',
           'last_updated','link_order','entity0_credit','entity1_credit'], **OPTS):
    f = chunk[chunk['entity0'].isin(recording_ids) | chunk['entity1'].isin(recording_ids)]
    if len(f): chunks.append(f)
l_rec_rec = pd.concat(chunks) if chunks else pd.DataFrame()
print(f"l_recording_recording → {len(l_rec_rec)} rows")

l_recording_recording → 30620 rows


In [9]:
release_group = filter_csv('./mbdump/mbdump/release_group',
    names=['id','gid','name','artist_credit','type','comment','edits_pending','last_updated'],
    filter_col='artist_credit', filter_ids=artist_credit_ids)
release_group_ids = set(release_group['id'])

release = filter_csv('./mbdump/mbdump/release',
    names=['id','gid','name','artist_credit','release_group','status',
           'packaging','language','script','barcode','comment',
           'edits_pending','quality','last_updated'],
    filter_col='release_group', filter_ids=release_group_ids)
release_ids = set(release['id'])

medium = filter_csv('./mbdump/mbdump/medium',
    names=['id','release','position','format','name',
           'edits_pending','last_updated','track_count','gid'],
    filter_col='release', filter_ids=release_ids)
medium_ids = set(medium['id'])

track = filter_csv('./mbdump/mbdump/track',
    names=['id','gid','recording','medium','position','number','name',
           'artist_credit','length','edits_pending','last_updated','is_data_track'],
    filter_col='medium', filter_ids=medium_ids)

print(f"release_group → {len(release_group)} | release → {len(release)} | medium → {len(medium)} | track → {len(track)}")

release_group → 722052 | release → 1184623 | medium → 1447371 | track → 17445540


In [10]:
SAVE = dict(sep='\t', index=False, header=False)

recording.to_csv(          f'{out}recording',           **SAVE)
artist.to_csv(             f'{out}artist',              **SAVE)
artist_credit.to_csv(      f'{out}artist_credit',       **SAVE)
artist_credit_name.to_csv( f'{out}artist_credit_name',  **SAVE)
recording_tag.to_csv(      f'{out}recording_tag',       **SAVE)
artist_tag.to_csv(         f'{out}artist_tag',          **SAVE)
tag.to_csv(                f'{out}tag',                 **SAVE)
l_rec_rec.to_csv(          f'{out}l_recording_recording', **SAVE)
release_group.to_csv(      f'{out}release_group',       **SAVE)
release.to_csv(            f'{out}release',             **SAVE)
medium.to_csv(             f'{out}medium',              **SAVE)
track.to_csv(              f'{out}track',               **SAVE)

for name, df in [('recording',recording),('artist',artist),
                 ('artist_credit',artist_credit),('artist_credit_name',artist_credit_name),
                 ('recording_tag',recording_tag),('artist_tag',artist_tag),
                 ('tag',tag),('l_rec_rec',l_rec_rec),
                 ('release_group',release_group),('release',release),
                 ('medium',medium),('track',track)]:
    print(f"{name:30s} → {len(df)} rows")

recording                      → 79359 rows
artist                         → 22973 rows
artist_credit                  → 23828 rows
artist_credit_name             → 35340 rows
recording_tag                  → 219438 rows
artist_tag                     → 74255 rows
tag                            → 9584 rows
l_rec_rec                      → 30620 rows
release_group                  → 722052 rows
release                        → 1184623 rows
medium                         → 1447371 rows
track                          → 17445540 rows
